# NB16: PTRC-HGSOC platinum resistance prediction

Within-cohort 5-fold stratified cross-validation on PTRC HGSOC
OpenSlideFM patient embeddings using the manuscript pipeline
(StandardScaler -> PCA(384) -> Ridge(alpha=30.0) -> Platt). Within
each fold the Ridge regressor uses the binary platinum-resistance
label as its regression target (treated as 0/1 float); the Platt
scaler maps the Ridge output to calibrated probabilities.

**Manuscript reference** (Methods + Table 2 + Discussion):
- Pipeline: StandardScaler -> PCA(k=384) -> Ridge(alpha=30.0) -> Platt
- Cohort: PTRC-HGSOC (Bergstrom et al.); n=158, 67 platinum-resistant,
  91 platinum-sensitive
- AUROC = 0.673 (95% CI: 0.588 - 0.757), AP = 0.631
- Operating point at Youden's J: threshold = 0.439, sensitivity = 61.2%,
  specificity = 72.5%, PPV = 62.1%, NPV = 71.7%
- Bootstrap reps: 2000 (external)

**Required env**: `WORKSPACE`, `PTRC_OSFM_FEATURES_PARQUET`,
`PTRC_LABELS_CSV`.
**Outputs**: `results/ptrc_platinum/{metrics.csv, predictions.csv,
operating_point.json, report.json}`.

In [ ]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
PTRC_OSFM_FEATURES_PARQUET = Path(os.environ["PTRC_OSFM_FEATURES_PARQUET"]).resolve()
PTRC_LABELS_CSV = Path(os.environ["PTRC_LABELS_CSV"]).resolve()
RESULTS_DIR = WORKSPACE / "results" / "ptrc_platinum"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
BOOT_N = 2000
N_FOLDS = 5

print(f"PTRC features : {PTRC_OSFM_FEATURES_PARQUET}")
print(f"PTRC labels   : {PTRC_LABELS_CSV}")
print(f"PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}, folds={N_FOLDS}, BOOT_N={BOOT_N}")


In [ ]:
X = pd.read_parquet(PTRC_OSFM_FEATURES_PARQUET)
if "patient" in X.columns:
    X = X.set_index("patient")
X.index = X.index.astype(str).str.upper().str.strip()

L = pd.read_csv(PTRC_LABELS_CSV)
L.columns = [str(c) for c in L.columns]
nmap = {re.sub(r"[^a-z0-9]+", "_", c.lower()): c for c in L.columns}

pat_col = nmap.get("patient") or nmap.get("patient_id") or nmap.get("case_id")
resp_col = (nmap.get("platinum_resistant") or nmap.get("resistant")
            or nmap.get("platinum_response") or nmap.get("response")
            or nmap.get("y") or nmap.get("label"))
assert pat_col is not None, f"need a patient column; got {list(L.columns)}"
assert resp_col is not None, f"need a platinum response column; got {list(L.columns)}"

L["_patient"] = L[pat_col].astype(str).str.upper().str.strip()
def to_resistant(v):
    s = str(v).strip().lower()
    if s in {"1", "yes", "true", "resistant", "r"}:
        return 1
    if s in {"0", "no", "false", "sensitive", "s"}:
        return 0
    try:
        return int(float(v))
    except Exception:
        return np.nan

L["_y"] = L[resp_col].map(to_resistant)
L = L.dropna(subset=["_patient", "_y"]).drop_duplicates("_patient").set_index("_patient")
L["_y"] = L["_y"].astype(int)

common = sorted(set(X.index) & set(L.index))
X = X.loc[common]
y = L.loc[common, "_y"].astype(int).values
print(f"aligned patients: {len(common)}  (manuscript ref: 158)")
print(f"resistant       : {int(y.sum())}  (manuscript ref: 67)")
print(f"sensitive       : {int(len(y) - y.sum())}  (manuscript ref: 91)")


In [ ]:
def fit_predict_fold(X_tr, X_te, y_tr_bin):
    pca_n = min(PCA_N, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("pca", PCA(n_components=pca_n, random_state=SEED)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
    ])
    pipe.fit(X_tr, y_tr_bin.astype(float))
    z_tr = pipe.predict(X_tr).reshape(-1, 1)
    platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z_tr, y_tr_bin)
    z_te = pipe.predict(X_te).reshape(-1, 1)
    return platt.predict_proba(z_te)[:, 1]

Xv = X.values.astype(np.float32)
n_splits = min(N_FOLDS, int(y.sum()), int(len(y) - y.sum()))
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
oof = np.zeros(len(y), dtype=float)
for tr, te in skf.split(Xv, y):
    oof[te] = fit_predict_fold(Xv[tr], Xv[te], y[tr])

print(f"CV complete; folds={n_splits}")


In [ ]:
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def ece(y, p, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    e = 0.0
    n = len(y)
    for i in range(n_bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < n_bins - 1 else p <= edges[i + 1])
        if m.sum() == 0:
            continue
        e += (m.sum() / n) * abs(y[m].mean() - p[m].mean())
    return float(e)

auc = float(roc_auc_score(y, oof))
ap = float(average_precision_score(y, oof))
br = float(brier_score_loss(y, oof))
auc_lo, auc_hi = boot_ci(y, oof, roc_auc_score)
ap_lo, ap_hi = boot_ci(y, oof, average_precision_score)

metrics = {
    "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
    "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi,
    "Brier": br, "ECE": ece(y, oof),
}
print(f"AUROC = {auc:.3f} ({auc_lo:.3f}-{auc_hi:.3f})  (manuscript ref: 0.673 [0.588-0.757])")
print(f"AP    = {ap:.3f} ({ap_lo:.3f}-{ap_hi:.3f})  (manuscript ref: 0.631)")
print(f"Brier = {br:.4f}  ECE = {metrics['ECE']:.4f}")


In [ ]:
# Youden's J operating point on Platt-calibrated CV scores
fpr, tpr, thr = roc_curve(y, oof)
youden = tpr - fpr
i = int(np.argmax(youden))
thr_youden = float(thr[i])
sens = float(tpr[i])
spec = float(1 - fpr[i])

pred = (oof >= thr_youden).astype(int)
tp = int(((pred == 1) & (y == 1)).sum())
fp = int(((pred == 1) & (y == 0)).sum())
fn = int(((pred == 0) & (y == 1)).sum())
tn = int(((pred == 0) & (y == 0)).sum())
ppv = tp / (tp + fp) if (tp + fp) else float("nan")
npv = tn / (tn + fn) if (tn + fn) else float("nan")

op = {
    "threshold_youden_J": thr_youden,
    "sensitivity": sens, "specificity": spec,
    "PPV": ppv, "NPV": npv,
    "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    "manuscript_refs": {
        "threshold": 0.439,
        "sensitivity": 0.612, "specificity": 0.725,
        "PPV": 0.621, "NPV": 0.717,
    },
}
print()
print(f"Youden J operating point: threshold = {thr_youden:.3f}  (ms ref 0.439)")
print(f"  sens = {sens:.3f}  (ms 0.612)")
print(f"  spec = {spec:.3f}  (ms 0.725)")
print(f"  PPV  = {ppv:.3f}  (ms 0.621)")
print(f"  NPV  = {npv:.3f}  (ms 0.717)")

(RESULTS_DIR / "operating_point.json").write_text(json.dumps(op, indent=2))


In [ ]:
pd.DataFrame([{"n": int(len(y)), "n_pos": int(y.sum()), **metrics}]).to_csv(
    RESULTS_DIR / "metrics.csv", index=False
)
pd.DataFrame({"patient": common, "y_true": y, "y_score": oof}).to_csv(
    RESULTS_DIR / "predictions.csv", index=False
)

report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA},
    "bootstrap_n": BOOT_N,
    "n_folds": int(n_splits),
    "n_total": int(len(y)),
    "n_resistant": int(y.sum()),
    "n_sensitive": int(len(y) - y.sum()),
    "metrics": metrics,
    "operating_point": op,
    "manuscript_refs": {
        "n_total": 158, "n_resistant": 67, "n_sensitive": 91,
        "AUROC": 0.673, "AUROC_CI": [0.588, 0.757], "AP": 0.631,
        "operating_threshold_youden": 0.439,
        "sens": 0.612, "spec": 0.725, "PPV": 0.621, "NPV": 0.717,
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
